# 05 Evaluation of the three LLM pipelines

Compares the explanation quality of:
* Pipeline 04a template baseline
* Pipeline 04b (JSON to Text): structured data as input
* Pipeline 04c (Vision to Text): waterfall plot as input
* Pipeline 04d (Tool Use): the LLM queries the data itself

Three evaluation layers:
1. Quantitative (no API key needed): length, token usage, cost, runtime
2. Faithfulness check (no API key): does the explanation mention the truly important features?
3. LLM as judge (API key needed): structured scoring on faithfulness, clarity, completeness

The judge is run by two models that are independent of the generation model (Sonnet):
Opus (primary) and OpenAI (cross vendor robustness). The same rubric is used for both.

In [1]:
from __future__ import annotations

import sys, json, re, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display

from utils import INSTANCE_IDS, RESULTS_DIR, EXPLANATIONS_DIR, PROMPTS_DIR
from utils.llm import ask_text, DEFAULT_MODEL, JUDGE_TEMPERATURE

LOSS_KEY = 'poisson_log'
MODEL    = DEFAULT_MODEL            # generation model (Sonnet); the judge uses Opus and OpenAI
PIPELINES = ['00', '04', '05', '06']
PIPELINE_LABELS = {
    '00': 'Template',
    '04': 'JSON to Text',
    '05': 'Vision',
    '06': 'Tool Use',
}
PIPELINE_COLORS = {
    'Template':     '#999999',
    'JSON to Text': '#4c72b0',
    'Vision':       '#dd8452',
    'Tool Use':     '#55a868',
}
XAI_MODELS = ['xgb', 'ebm']

# Cost per 1M tokens (claude-sonnet-4-6, generation model).
COST_INPUT_PER_M       = 3.00   # USD, regular input tokens
COST_CACHE_READ_PER_M  = 0.30   # USD, cache read tokens (90 percent discount)
COST_OUTPUT_PER_M      = 15.00  # USD, output tokens

print('Setup complete.')
print(f'JUDGE_TEMPERATURE = {JUDGE_TEMPERATURE}  (deterministic)')

Setup complete.
JUDGE_TEMPERATURE = 0.0  (deterministic)


## 1. Load results

In [2]:
records = []
for pipeline in PIPELINES:
    p = RESULTS_DIR / f'pipeline{pipeline}'
    for xai in XAI_MODELS:
        for iid in INSTANCE_IDS:
            f = p / f'{xai}_inst{iid}.json'
            if not f.exists():
                print(f'MISSING: {f}')
                continue
            d = json.loads(f.read_text())
            usage = d.get('usage', {})
            in_tok   = usage.get('input_tokens', 0)
            out_tok  = usage.get('output_tokens', 0)
            cache_r  = usage.get('cache_read_input_tokens', 0)
            # Cache read tokens cost only 0.30/M instead of 3.00/M
            regular_in = max(in_tok - cache_r, 0)
            cost = (
                regular_in * COST_INPUT_PER_M
                + cache_r  * COST_CACHE_READ_PER_M
                + out_tok  * COST_OUTPUT_PER_M
            ) / 1_000_000
            records.append({
                'pipeline':      pipeline,
                'pipeline_label': PIPELINE_LABELS[pipeline],
                'xai_model':     xai.upper(),
                'instance_id':   iid,
                'explanation':   d.get('explanation', ''),
                'word_count':    len(d.get('explanation', '').split()),
                'tok_input':     in_tok,
                'tok_output':    out_tok,
                'tok_cache':     cache_r,
                'tok_total':     in_tok + out_tok,
                'cost_usd':      round(cost, 5),
                'elapsed_s':     d.get('elapsed_s', 0),
                'n_tool_calls':  d.get('n_tool_calls', 0),
                'y_true':        d.get('y_true', None),
                'prediction':    d.get('prediction', None),
            })

df = pd.DataFrame(records)
print(f'{len(df)} results loaded.')
display(df.groupby(['pipeline_label', 'xai_model']).size().unstack())

80 results loaded.


xai_model,EBM,XGB
pipeline_label,,
JSON to Text,10,10
Template,10,10
Tool Use,10,10
Vision,10,10


In [3]:
# Document the instance sample.
# Stratified selection over cnt quintiles (no extreme cases: cnt >= 31, no thunderstorm, no holiday cluster).
instance_doc = [
    {"instance_id": 224,  "cnt_true":  51, "note": "Thu month02 13h, clear, ~8 C, 2011"},
    {"instance_id": 580,  "cnt_true": 159, "note": "Sun month03 17h, clear, ~13 C, 2011"},
    {"instance_id": 1041, "cnt_true": 251, "note": "Sun month05 10h, cloudy, ~27 C, 2011"},
    {"instance_id": 1481, "cnt_true": 114, "note": "Sat month07 08h, clear, ~32 C, 2011"},
    {"instance_id": 1677, "cnt_true": 410, "note": "Fri month08 18h, cloudy, ~30 C, 2011"},
    {"instance_id": 2058, "cnt_true":  31, "note": "Fri month10 05h, clear, ~14 C, 2011"},
    {"instance_id": 2510, "cnt_true":  89, "note": "Wed month12 10h, cloudy, ~20 C, 2011"},
    {"instance_id": 3543, "cnt_true": 191, "note": "Sun month05 09h, cloudy, ~21 C, 2012"},
    {"instance_id": 3847, "cnt_true": 302, "note": "Sun month06 20h, clear, ~25 C, 2012"},
    {"instance_id": 4454, "cnt_true": 557, "note": "Wed month09 07h, clear, ~21 C, 2012"},
]
inst_df = pd.DataFrame(instance_doc)
print("Selected test instances (stratified quantile sampling):")
print("  cnt range: 31 to 557, spread over 5 cnt quintiles")
print("  No cnt <= 20, no thunderstorm (weathersit=4), both years (2011/2012)")
print()
display(inst_df)

Selected test instances (stratified quantile sampling):
  cnt range: 31 to 557, spread over 5 cnt quintiles
  No cnt <= 20, no thunderstorm (weathersit=4), both years (2011/2012)



,instance_id,cnt_true,note
0,224,51,"Thu month02 13h, clear, ~8 C, 2011"
1,580,159,"Sun month03 17h, clear, ~13 C, 2011"
2,1041,251,"Sun month05 10h, cloudy, ~27 C, 2011"
3,1481,114,"Sat month07 08h, clear, ~32 C, 2011"
4,1677,410,"Fri month08 18h, cloudy, ~30 C, 2011"
5,2058,31,"Fri month10 05h, clear, ~14 C, 2011"
6,2510,89,"Wed month12 10h, cloudy, ~20 C, 2011"
7,3543,191,"Sun month05 09h, cloudy, ~21 C, 2012"
8,3847,302,"Sun month06 20h, clear, ~25 C, 2012"
9,4454,557,"Wed month09 07h, clear, ~21 C, 2012"


## 2. Quantitative analysis

In [4]:
quant = df.groupby('pipeline_label').agg(
    words_mean   =('word_count',  'mean'),
    words_std    =('word_count',  'std'),
    tok_input    =('tok_input',   'mean'),
    tok_output   =('tok_output',  'mean'),
    tok_total    =('tok_total',   'mean'),
    cost_usd_tot =('cost_usd',    'sum'),
    time_s       =('elapsed_s',   'mean'),
    tool_calls   =('n_tool_calls','mean'),
).round(2)
print('Quantitative comparison (mean per pipeline):')
display(quant)

Quantitative comparison (mean per pipeline):


,words_mean,words_std,tok_input,tok_output,tok_total,cost_usd_tot,time_s,tool_calls
pipeline_label,,,,,,,,
JSON to Text,249.6,13.95,599.55,523.80,1123.35,0.17,11.49,0.0
Template,56.9,1.77,0.00,0.00,0.00,0.00,0.00,0.0
Tool Use,403.0,23.72,5786.30,1268.30,7054.60,0.73,33.01,6.2
Vision,238.9,15.34,1084.90,571.45,1656.35,0.19,13.15,0.0


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
labels = [PIPELINE_LABELS[p] for p in PIPELINES]
colors = [PIPELINE_COLORS[l] for l in labels]

# Word count
vals = [df[df.pipeline == p]['word_count'].mean() for p in PIPELINES]
axes[0].bar(labels, vals, color=colors)
axes[0].set_title('Mean word count')
axes[0].set_ylabel('Words')

# Token usage
in_vals  = [df[df.pipeline == p]['tok_input'].mean()  for p in PIPELINES]
out_vals = [df[df.pipeline == p]['tok_output'].mean() for p in PIPELINES]
x = range(len(labels))
axes[1].bar(x, in_vals,  label='Input',  color=colors, alpha=0.5)
axes[1].bar(x, out_vals, label='Output', color=colors, bottom=in_vals)
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(labels)
axes[1].set_title('Mean token usage')
axes[1].set_ylabel('Tokens')
axes[1].legend()

# Total cost
vals = [df[df.pipeline == p]['cost_usd'].sum() for p in PIPELINES]
axes[2].bar(labels, vals, color=colors)
axes[2].set_title('Total cost')
axes[2].set_ylabel('USD')

plt.suptitle('Quantitative comparison of the four pipelines (incl. template baseline)', y=1.02)
plt.tight_layout()
out_path = RESULTS_DIR / 'eval_quantitative.png'
plt.savefig(out_path, dpi=130, bbox_inches='tight')
display(fig)
print(f'Saved: {out_path}')

<Figure size 1300x400 with 3 Axes>

Saved: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/eval_quantitative.png


## 3. LLM as judge evaluation

An independent judge model scores each explanation on three dimensions (1 to 5):
* Faithfulness: are the most important drivers described correctly?
* Clarity: is the explanation understandable for non experts?
* Completeness: are prediction, drivers and a practical implication covered?

The rubric lives in prompts/judge_system.md and already passes the top 3 SHAP
contributions as ground truth, so the judge compares against the real drivers.

In [6]:
JUDGE_SYSTEM = (PROMPTS_DIR / "judge_system.md").read_text()


WEEKDAYS_JUDGE = {0:"Sunday", 1:"Monday", 2:"Tuesday", 3:"Wednesday",
                  4:"Thursday", 5:"Friday", 6:"Saturday"}
MONTHS_JUDGE   = {1:"January", 2:"February", 3:"March", 4:"April", 5:"May",
                  6:"June", 7:"July", 8:"August", 9:"September",
                  10:"October", 11:"November", 12:"December"}
WEATHER_JUDGE  = {1:"clear/few clouds", 2:"mist/cloudy",
                  3:"light rain/snow", 4:"heavy rain/thunderstorm"}


def build_judge_prompt(row: dict, xai_model: str, instance_id: int) -> str:
    local_path = EXPLANATIONS_DIR / f"local_{xai_model.lower()}_{LOSS_KEY}_inst{instance_id}.json"
    l = json.loads(local_path.read_text())
    fv = l["feature_values"]

    top3 = [{"feature": c["feature"], "contribution": c["contribution"],
              "value": c["value"]}
             for c in l["contributions"][:3]]

    # Human readable feature values for the judge.
    fv_readable = {
        "time":         f"{int(fv['hr']):02d}:00",
        "weekday":      WEEKDAYS_JUDGE.get(int(fv["weekday"]), str(fv["weekday"])),
        "month":        MONTHS_JUDGE.get(int(fv["mnth"]), str(fv["mnth"])),
        "year":         "2011" if int(fv["yr"]) == 0 else "2012",
        "weather":      WEATHER_JUDGE.get(int(fv["weathersit"]), str(fv["weathersit"])),
        "temperature_celsius": f"~{float(fv['temp']) * 41:.1f} C",
        "humidity":     f"{float(fv['hum']) * 100:.0f} %",
        "wind_speed":   f"{float(fv['windspeed']) * 67:.1f} km/h",
        "holiday":      "yes" if int(fv["holiday"]) == 1 else "no",
    }

    ground_truth = {
        "model":                  xai_model,
        "prediction":             l["prediction"],
        "y_true":                 l["y_true"],
        "top3_drivers":           top3,
        "feature_values_readable": fv_readable,
    }

    # For Tool Use: embed the real tool call trace so the judge can verify the numbers.
    pipeline = row.get("pipeline", "")
    if "06" in pipeline or pipeline == "06_tooluse":
        trace_path = RESULTS_DIR / f"pipeline06/{xai_model.lower()}_inst{instance_id}.json"
        if trace_path.exists():
            trace_data = json.loads(trace_path.read_text())
            ground_truth["tool_call_trace"] = [
                {
                    "round":     i + 1,
                    "tool":      c["tool"],
                    "arguments": c["arguments"],
                    "result":    c["result_preview"],
                }
                for i, c in enumerate(trace_data.get("tool_calls", []))
            ]
            ground_truth["tool_trace_note"] = (
                "The retrieved values (contributions, percentiles, counterfactuals) "
                "are correct and may be counted as evidence for faithfulness."
            )

    output_instruction = (
        "Answer only in the XML format from the system prompt.\n"
        "Per criterion: first the reasoning (1 to 2 sentences), then the score as an XML tag.\n"
        "\n"
        "<faithfulness_reasoning>Choose anchor point, check deductions, compute final score</faithfulness_reasoning>\n"
        "<faithfulness>N</faithfulness>\n"
        "<clarity_reasoning>Choose anchor point, check deductions, compute final score</clarity_reasoning>\n"
        "<clarity>N</clarity>\n"
        "<completeness_reasoning>Choose anchor point, check deductions, compute final score</completeness_reasoning>\n"
        "<completeness>N</completeness>"
    )
    return json.dumps({
        "task": (
            "Score the following explanation using the defined rubric. "
            "Give a score (1 to 5) for each criterion and justify briefly."
        ),
        "ground_truth": ground_truth,
        "explanation": row["explanation"],
        "output_format": output_instruction,
    }, ensure_ascii=False, indent=2)

In [7]:
# Primary judge: Opus (claude-opus-4-8), independent of the generation model (Sonnet).
# The same rubric is also run by the OpenAI judge in section 8 for cross vendor robustness.
from utils.judge import parse_judge_response as _parse_judge_response

OPUS_MODEL = 'claude-opus-4-8'
_MAX_JUDGE = 900

opus_path = RESULTS_DIR / 'eval_llm_judge_opus.json'
_opus_cached = json.loads(opus_path.read_text()) if opus_path.exists() else []

if _opus_cached and len(_opus_cached) == len(df):
    judge_df = pd.DataFrame(_opus_cached)
    for col in ['faithfulness', 'clarity', 'completeness']:
        judge_df[col] = pd.to_numeric(judge_df[col], errors='coerce')
    print(f'Opus judge results loaded from cache: {len(judge_df)} entries')
    print('(delete the cache file to recompute)')
else:
    if _opus_cached:
        print(f'Cache has {len(_opus_cached)} entries, df has {len(df)}. Recomputing ...')
    else:
        print(f'Running Opus judge ({OPUS_MODEL}) for all {len(df)} explanations ...')
    _rows = []
    total_in, total_out = 0, 0
    MAX_RETRIES = 3
    for _, row in df.iterrows():
        prompt = build_judge_prompt(row.to_dict(), row['xai_model'], row['instance_id'])
        scores, in_tok, out_tok, raw = {}, 0, 0, ''
        for attempt in range(MAX_RETRIES):
            response = ask_text(
                prompt,
                system=JUDGE_SYSTEM,
                model=OPUS_MODEL,
                max_tokens=_MAX_JUDGE,
                cache_system=True,
                temperature=JUDGE_TEMPERATURE,   # gated out for Opus inside ask_text
            )
            usage  = response.get('usage', {})
            in_tok = usage.get('input_tokens', 0)
            out_tok = usage.get('output_tokens', 0)
            raw    = response['content'][0]['text'].strip()
            scores = _parse_judge_response(raw)
            if all(scores.get(k) is not None for k in ['faithfulness', 'clarity', 'completeness']):
                break
            print(f'Retry {attempt+1}/{MAX_RETRIES}: {row["pipeline_label"]} {row["xai_model"]} inst={row["instance_id"]}')
        total_in += in_tok; total_out += out_tok
        _rows.append({
            'pipeline_label': row['pipeline_label'],
            'xai_model':      row['xai_model'],
            'instance_id':    row['instance_id'],
            'faithfulness':   scores.get('faithfulness', None),
            'clarity':        scores.get('clarity', None),
            'completeness':   scores.get('completeness', None),
            'reasoning': {
                'faithfulness':  scores.get('faithfulness_reasoning', ''),
                'clarity':       scores.get('clarity_reasoning', ''),
                'completeness':  scores.get('completeness_reasoning', ''),
            },
            'raw_response': raw,
        })
        print(f'  {row["pipeline_label"]:12s} {row["xai_model"]} inst={row["instance_id"]:4d}  '
              f'F={scores.get("faithfulness","?")} '
              f'C={scores.get("clarity","?")} '
              f'Co={scores.get("completeness","?")}  in={in_tok}')
    judge_df = pd.DataFrame(_rows)
    opus_path.write_text(json.dumps(_rows, indent=2, ensure_ascii=False))
    print(f'\nTotal: input={total_in}  output={total_out}')

# Opus is the primary judge. Alias kept so the statistics cells can read judge_opus_df.
judge_opus_df = judge_df

Running Opus judge (claude-opus-4-8) for all 80 explanations ...
  Template     XGB inst= 224  F=5 C=4 Co=5  in=773
  Template     XGB inst= 580  F=5 C=4 Co=5  in=768
  Template     XGB inst=1041  F=5 C=4 Co=5  in=769
  Template     XGB inst=1481  F=5 C=4 Co=5  in=770
  Template     XGB inst=1677  F=5 C=4 Co=4  in=770
  Template     XGB inst=2058  F=5 C=4 Co=5  in=772
  Template     XGB inst=2510  F=5 C=4 Co=5  in=769
  Template     XGB inst=3543  F=5 C=5 Co=4  in=764
  Template     XGB inst=3847  F=5 C=5 Co=5  in=772
  Template     XGB inst=4454  F=5 C=4 Co=5  in=769
  Template     EBM inst= 224  F=5 C=4 Co=5  in=773
  Template     EBM inst= 580  F=5 C=4 Co=5  in=773
  Template     EBM inst=1041  F=5 C=4 Co=5  in=769
  Template     EBM inst=1481  F=5 C=4 Co=4  in=770
  Template     EBM inst=1677  F=5 C=4 Co=4  in=770
  Template     EBM inst=2058  F=5 C=4 Co=5  in=770
  Template     EBM inst=2510  F=5 C=4 Co=5  in=769
  Template     EBM inst=3543  F=5 C=4 Co=5  in=764
  Template     EB

## 4. Judge results

In [8]:
# Document missing scores (JSON parse failures in the judge output)
null_mask = judge_df[['faithfulness', 'clarity', 'completeness']].isnull().any(axis=1)
if null_mask.any():
    print(f"WARNING: {null_mask.sum()} entries with None scores (JSON parsing failed):")
    display(judge_df[null_mask][['pipeline_label', 'xai_model', 'instance_id']])
    print("   These entries are excluded by .mean() automatically (pandas NaN handling).")
    print()

judge_summary = judge_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].mean().round(2)
judge_counts  = judge_df.groupby('pipeline_label')[['faithfulness']].count().rename(
    columns={'faithfulness': 'n_valid'}
)
judge_summary = judge_summary.join(judge_counts)
print('Mean judge scores (1 to 5) per pipeline  [None entries excluded from the mean]:')
display(judge_summary)

print()
judge_xai = judge_df.groupby(['pipeline_label', 'xai_model'])[['faithfulness', 'clarity', 'completeness']].mean().round(2)
print('Broken down by model:')
display(judge_xai)

Mean judge scores (1 to 5) per pipeline  [None entries excluded from the mean]:


,faithfulness,clarity,completeness,n_valid
pipeline_label,,,,
JSON to Text,5.0,4.10,5.0,20
Template,5.0,4.15,4.8,20
Tool Use,5.0,4.10,5.0,20
Vision,4.5,4.00,5.0,20



Broken down by model:


faithfulness  clarity  completeness
pipeline_label xai_model                                     
JSON to Text   EBM                 5.0      4.1           5.0
               XGB                 5.0      4.1           5.0
Template       EBM                 5.0      4.1           4.8
               XGB                 5.0      4.2           4.8
Tool Use       EBM                 5.0      4.1           5.0
               XGB                 5.0      4.1           5.0
Vision         EBM                 4.2      4.0           5.0
               XGB                 4.8      4.0           5.0

In [9]:
# Radar chart
import numpy as np

criteria   = ['Faithfulness', 'Clarity', 'Completeness']
n          = len(criteria)
angles     = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
angles    += angles[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

for label, color in PIPELINE_COLORS.items():
    row = judge_summary.loc[label] if label in judge_summary.index else None
    if row is None: continue
    values = [row['faithfulness'], row['clarity'], row['completeness']]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=label, color=color)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(criteria, fontsize=12)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(['1','2','3','4','5'], fontsize=8)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.set_title('LLM judge scores (1 to 5) per pipeline', pad=20)

plt.tight_layout()
out_path = RESULTS_DIR / 'eval_radar.png'
plt.savefig(out_path, dpi=130, bbox_inches='tight')
display(fig)
print(f'Saved: {out_path}')

<Figure size 600x600 with 1 Axes>

Saved: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/eval_radar.png


## 5. Overview and recommendation

In [10]:
# Combined table: quantitative + LLM judge
quant_p = df.groupby('pipeline_label').agg(
    words      =('word_count', 'mean'),
    tokens_in  =('tok_input',  'mean'),
    tokens_out =('tok_output', 'mean'),
    cost_usd   =('cost_usd',   'sum'),
    time_s     =('elapsed_s',  'mean'),
    tool_calls =('n_tool_calls','mean'),
).round(2)

combined = quant_p.copy()
if len(judge_df) > 0:
    judge_p = judge_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].mean().round(2)
    judge_p.columns = ['Judge_Faith', 'Judge_Clarity', 'Judge_Complete']
    judge_std = judge_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].std().round(3)
    judge_std.columns = ['Judge_Faith_std', 'Judge_Clarity_std', 'Judge_Complete_std']
    judge_n = judge_df.groupby('pipeline_label')[['faithfulness']].count().rename(
        columns={'faithfulness': 'Judge_n'}
    )
    combined = combined.join(judge_p).join(judge_std).join(judge_n)

print('Overview:')
display(combined)

combined.to_csv(RESULTS_DIR / 'eval_summary.csv')
print(f'Saved: {RESULTS_DIR / "eval_summary.csv"}')

Overview:


,words,tokens_in,tokens_out,cost_usd,time_s,tool_calls,Judge_Faith,Judge_Clarity,Judge_Complete,Judge_Faith_std,Judge_Clarity_std,Judge_Complete_std,Judge_n
pipeline_label,,,,,,,,,,,,,
JSON to Text,249.6,599.55,523.80,0.17,11.49,0.0,5.0,4.10,5.0,0.000,0.308,0.00,20
Template,56.9,0.00,0.00,0.00,0.00,0.0,5.0,4.15,4.8,0.000,0.366,0.41,20
Tool Use,403.0,5786.30,1268.30,0.73,33.01,6.2,5.0,4.10,5.0,0.000,0.447,0.00,20
Vision,238.9,1084.90,571.45,0.19,13.15,0.0,4.5,4.00,5.0,0.688,0.324,0.00,20


Saved: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/eval_summary.csv


In [11]:
print('=== RECOMMENDATION ===')
print()

if len(judge_df) > 0:
    judge_total = judge_df.groupby('pipeline_label')[['faithfulness','clarity','completeness']].mean().sum(axis=1)
    best_pipeline = judge_total.idxmax()
    print(f'Best pipeline by total LLM judge score: {best_pipeline}')
    print()
    print('Score ranking:')
    for label, score in judge_total.sort_values(ascending=False).items():
        print(f'  {label:12s}: {score:.2f} / 15')
    print()

print('Trade off analysis:')
print('  JSON to Text : lowest token usage, good faithfulness (numbers directly available)')
print('  Vision       : medium usage, the LLM reads contributions from the plot (can be fuzzy)')
print('  Tool Use     : highest usage, the LLM drives the query itself, potentially the best')
print('                 completeness because it can probe on demand')

=== RECOMMENDATION ===

Best pipeline by total LLM judge score: JSON to Text

Score ranking:
  JSON to Text: 14.10 / 15
  Tool Use    : 14.10 / 15
  Template    : 13.95 / 15
  Vision      : 13.50 / 15

Trade off analysis:
  JSON to Text : lowest token usage, good faithfulness (numbers directly available)
  Vision       : medium usage, the LLM reads contributions from the plot (can be fuzzy)
  Tool Use     : highest usage, the LLM drives the query itself, potentially the best
                 completeness because it can probe on demand


## 6. Methodological limitations

The following points should be kept in mind when interpreting the results:

| # | Limitation | Consequence |
|---|---|---|
| 1 | Ceiling effect: the judge almost only assigns 4 to 5 out of 5 | Clarity and completeness do not discriminate between pipelines. The radar chart suggests more differentiation than is present |
| 2 | Independent judges: both judge models (Opus, OpenAI) differ from the generation model (Sonnet) | Self preference bias is avoided by design. Opus is the primary judge, OpenAI is the cross vendor robustness check |
| 3 | n = 10 to 20 per pipeline | Low statistical power. Mean differences between pipelines cannot be confirmed by inference statistics |
| 4 | No repeated sampling | LLM variability is not measured. A consistency metric is missing |
| 5 | Random train/test split on time series data | Possible temporal leakage. R squared above 0.95 is partly explained by this. Acceptable for an XAI demonstration, but generalisation claims are limited |
| 6 | Selection bias of the Ichmoukhamedov metrics (see notebook 06) | RA/SA/VA are computed only over features the LLM itself mentioned. Unmentioned top K features are not penalised, so the metric overstates real faithfulness |

Conclusion:
The results are an exploratory demonstration. Statistically robust statements about
pipeline differences require larger samples (n >= 30 per pipeline), an independent judge
and repeated sampling.

In [12]:
# Quantify the ceiling effect: how many scores are = 5?
if len(judge_df) > 0:
    total_valid = judge_df[['faithfulness', 'clarity', 'completeness']].count().sum()
    total_5     = (judge_df[['faithfulness', 'clarity', 'completeness']] == 5).sum().sum()
    total_4     = (judge_df[['faithfulness', 'clarity', 'completeness']] == 4).sum().sum()
    print(f"Ceiling effect: {total_5}/{total_valid} scores = 5 ({100*total_5/total_valid:.0f}%)")
    print(f"               {total_4}/{total_valid} scores = 4 ({100*total_4/total_valid:.0f}%)")
    print(f"               {total_valid - total_4 - total_5} scores <= 3")
    print()
    print("Standard deviations of the judge scores per pipeline:")
    display(
        judge_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']]
        .std().round(3).rename(columns=lambda c: c + '_std')
    )

Ceiling effect: 157/240 scores = 5 (65%)
               79/240 scores = 4 (33%)
               4 scores <= 3

Standard deviations of the judge scores per pipeline:


,faithfulness_std,clarity_std,completeness_std
pipeline_label,,,
JSON to Text,0.000,0.308,0.00
Template,0.000,0.366,0.41
Tool Use,0.000,0.447,0.00
Vision,0.688,0.324,0.00


## 7. Inference statistics: bootstrap CI, Wilcoxon signed rank, Cliff's delta

Basis: Opus judge (calibrated prompt, independent model), n = 20 paired observations per
pipeline (10 instances x 2 XAI models, consistent instance IDs across all pipelines).

All functions are parameterised for arbitrary n and apply directly to a larger sample later.

Rule: from here on every mean statement about pipeline differences is reported only with a
95 percent bootstrap CI and a Wilcoxon p value.

In [13]:
from utils.stats import bootstrap_ci, cliffs_delta, _delta_magnitude, wilcoxon_pairwise

print('Stat functions loaded: bootstrap_ci / cliffs_delta / wilcoxon_pairwise')

Stat functions loaded: bootstrap_ci / cliffs_delta / wilcoxon_pairwise


In [14]:
# Bootstrap CI for all mean statements (Opus judge)
STAT_PIPELINES = list(PIPELINE_COLORS.keys())   # ['Template', 'JSON to Text', 'Vision', 'Tool Use']
STAT_METRICS   = ['faithfulness', 'clarity', 'completeness']

ci_rows = []
for pipeline in STAT_PIPELINES:
    row = {'pipeline': pipeline}
    subset = judge_opus_df[judge_opus_df['pipeline_label'] == pipeline]
    for metric in STAT_METRICS:
        lo, hi, obs = bootstrap_ci(subset[metric].values)
        row[f'{metric}_mean']   = round(obs, 3)
        row[f'{metric}_ci_lo']  = round(lo, 3)
        row[f'{metric}_ci_hi']  = round(hi, 3)
        row[f'{metric}_ci_str'] = f'{obs:.2f} [{lo:.2f}, {hi:.2f}]'
    ci_rows.append(row)

ci_df = pd.DataFrame(ci_rows).set_index('pipeline')

print('95 percent bootstrap CI of all means, Opus judge  (n = 20 per pipeline, 2000 resamples)')
print('Format: mean [CI_lower, CI_upper]\n')
display(ci_df[[f'{m}_ci_str' for m in STAT_METRICS]].rename(
    columns={f'{m}_ci_str': m.capitalize() for m in STAT_METRICS}
))

95 percent bootstrap CI of all means, Opus judge  (n = 20 per pipeline, 2000 resamples)
Format: mean [CI_lower, CI_upper]



,Faithfulness,Clarity,Completeness
pipeline,,,
Template,"5.00 [5.00, 5.00]","4.15 [4.00, 4.30]","4.80 [4.60, 4.95]"
JSON to Text,"5.00 [5.00, 5.00]","4.10 [4.00, 4.25]","5.00 [5.00, 5.00]"
Vision,"4.50 [4.20, 4.80]","4.00 [3.85, 4.15]","5.00 [5.00, 5.00]"
Tool Use,"5.00 [5.00, 5.00]","4.10 [3.90, 4.30]","5.00 [5.00, 5.00]"


In [15]:
# Pairwise Wilcoxon signed rank + Cliff's delta (Opus judge)
# Paired over (instance_id, xai_model): consistent IDs across all pipelines
print("Pairwise Wilcoxon signed rank (two sided) + Cliff's delta, Opus judge")
print('Significance level alpha = 0.05  |  effect size: negligible / small / medium / large\n')

for metric in STAT_METRICS:
    wdf = wilcoxon_pairwise(judge_opus_df, STAT_PIPELINES, metric)
    print(f'-- {metric.upper()} --')
    display(wdf[['pipeline_a', 'pipeline_b', 'n_pairs', 'mean_a', 'mean_b',
                 'delta_mean', 'p_value', 'cliffs_d', 'magnitude']])
    print()

Pairwise Wilcoxon signed rank (two sided) + Cliff's delta, Opus judge
Significance level alpha = 0.05  |  effect size: negligible / small / medium / large

-- FAITHFULNESS --


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


,pipeline_a,pipeline_b,n_pairs,mean_a,mean_b,delta_mean,p_value,cliffs_d,magnitude
0,Template,JSON to Text,20,5.0,5.0,0.0,NaN,0.0,negligible
1,Template,Vision,20,5.0,4.5,0.5,0.0083,0.4,medium
2,Template,Tool Use,20,5.0,5.0,0.0,NaN,0.0,negligible
3,JSON to Text,Vision,20,5.0,4.5,0.5,0.0083,0.4,medium
4,JSON to Text,Tool Use,20,5.0,5.0,0.0,NaN,0.0,negligible
5,Vision,Tool Use,20,4.5,5.0,-0.5,0.0083,-0.4,medium



-- CLARITY --


,pipeline_a,pipeline_b,n_pairs,mean_a,mean_b,delta_mean,p_value,cliffs_d,magnitude
0,Template,JSON to Text,20,4.15,4.1,0.05,0.5637,0.050,negligible
1,Template,Vision,20,4.15,4.0,0.15,0.1797,0.142,negligible
2,Template,Tool Use,20,4.15,4.1,0.05,0.6547,0.043,negligible
3,JSON to Text,Vision,20,4.10,4.0,0.10,0.3173,0.095,negligible
4,JSON to Text,Tool Use,20,4.10,4.1,0.00,1.0000,-0.005,negligible
5,Vision,Tool Use,20,4.00,4.1,-0.10,0.3173,-0.095,negligible



-- COMPLETENESS --


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


,pipeline_a,pipeline_b,n_pairs,mean_a,mean_b,delta_mean,p_value,cliffs_d,magnitude
0,Template,JSON to Text,20,4.8,5.0,-0.2,0.0455,-0.2,small
1,Template,Vision,20,4.8,5.0,-0.2,0.0455,-0.2,small
2,Template,Tool Use,20,4.8,5.0,-0.2,0.0455,-0.2,small
3,JSON to Text,Vision,20,5.0,5.0,0.0,NaN,0.0,negligible
4,JSON to Text,Tool Use,20,5.0,5.0,0.0,NaN,0.0,negligible
5,Vision,Tool Use,20,5.0,5.0,0.0,NaN,0.0,negligible


In [16]:
# Summary table with CI + p value for each core statement. Saved to results/.
stat_summary_rows = []
for pipeline in STAT_PIPELINES:
    subset = judge_opus_df[judge_opus_df['pipeline_label'] == pipeline]
    entry  = {'pipeline': pipeline, 'n': len(subset)}
    for metric in STAT_METRICS:
        lo, hi, obs = bootstrap_ci(subset[metric].values)
        entry[f'{metric}_mean']  = round(obs,  3)
        entry[f'{metric}_ci_lo'] = round(lo,   3)
        entry[f'{metric}_ci_hi'] = round(hi,   3)
    stat_summary_rows.append(entry)

stat_summary_df = pd.DataFrame(stat_summary_rows)
out_stat = RESULTS_DIR / 'eval_stat_summary.csv'
stat_summary_df.to_csv(out_stat, index=False)
print(f'Saved: {out_stat}')

# Merge the Wilcoxon results of all metrics
all_wilcoxon = []
for metric in STAT_METRICS:
    wdf = wilcoxon_pairwise(judge_opus_df, STAT_PIPELINES, metric)
    wdf.insert(0, 'metric', metric)
    all_wilcoxon.append(wdf)
wilcoxon_all_df = pd.concat(all_wilcoxon, ignore_index=True)
out_wil = RESULTS_DIR / 'eval_wilcoxon_cliffsdelta.csv'
wilcoxon_all_df.to_csv(out_wil, index=False)
print(f'Saved: {out_wil}')

print()
print('=== STATISTICALLY SUPPORTED CORE STATEMENTS (Opus judge, n=20) ===')
print()
for metric in STAT_METRICS:
    print(f'  {metric.upper()}:')
    subset_ci = ci_df
    for pipeline in STAT_PIPELINES:
        lo  = subset_ci.loc[pipeline, f'{metric}_ci_lo']
        hi  = subset_ci.loc[pipeline, f'{metric}_ci_hi']
        obs = subset_ci.loc[pipeline, f'{metric}_mean']
        print(f'    {pipeline:<12s}: {obs:.2f}  95% CI [{lo:.2f}, {hi:.2f}]')
    print()
print('Note: overlapping CIs and p > 0.05 (see Wilcoxon table) show that')
print('most differences cannot be confirmed by inference statistics at n=20.')
print('Robust statements need a larger sample (n approx 200).')

Saved: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/eval_stat_summary.csv
Saved: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/eval_wilcoxon_cliffsdelta.csv

=== STATISTICALLY SUPPORTED CORE STATEMENTS (Opus judge, n=20) ===

  FAITHFULNESS:
    Template    : 5.00  95% CI [5.00, 5.00]
    JSON to Text: 5.00  95% CI [5.00, 5.00]
    Vision      : 4.50  95% CI [4.20, 4.80]
    Tool Use    : 5.00  95% CI [5.00, 5.00]

  CLARITY:
    Template    : 4.15  95% CI [4.00, 4.30]
    JSON to Text: 4.10  95% CI [4.00, 4.25]
    Vision      : 4.00  95% CI [3.85, 4.15]
    Tool Use    : 4.10  95% CI [3.90, 4.30]

  COMPLETENESS:
    Template    : 4.80  95% CI [4.60, 4.95]
    JSON to Text: 5.00  95% CI [5.00, 5.00]
    Vision      : 5.00  95% CI [5.00, 5.00]
    Tool Use    : 5.00  95% CI [5.00, 5.00]

Note: overlapping CIs and p > 0.05 (see Wilcoxon table) show that
most differences cannot be confirmed by inference statistics at n=20.

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se
/Library/Frameworks/Python.fra

## 8. OpenAI cross vendor judge

The same rubric is run by an OpenAI model. If the pipeline ranking stays the same under a
different vendor, the ranking is robust and not an artefact of one model family.

In [17]:
from utils.llm import ask_openai_text, OPENAI_JUDGE_MODEL_TEST

# Configuration
OPENAI_JUDGE_MODEL = OPENAI_JUDGE_MODEL_TEST   # for the final run use OPENAI_JUDGE_MODEL_FINAL

# Free tier (tier 0): 3 RPM means a 20 s pause. Tier 1 and above: 0 s.
REQUEST_DELAY_S = 20

OPENAI_CACHE_PATH = RESULTS_DIR / 'eval_llm_judge_openai.json'

# Cost gpt-4o-mini (USD per 1M tokens)
OAI_COST_IN_PER_M  = 0.15
OAI_COST_OUT_PER_M = 0.60

_oai_cached = json.loads(OPENAI_CACHE_PATH.read_text()) if OPENAI_CACHE_PATH.exists() else []

if _oai_cached and len(_oai_cached) == len(df):
    print(f'OpenAI judge results loaded from cache: {OPENAI_CACHE_PATH.name}')
    print('(delete the cache file to recompute)')
else:
    if _oai_cached:
        print(f'Cache has {len(_oai_cached)} entries, df has {len(df)}. Recomputing ...')
    else:
        est_cost = len(df) * (1500 * OAI_COST_IN_PER_M + 300 * OAI_COST_OUT_PER_M) / 1_000_000
        est_min  = len(df) * REQUEST_DELAY_S / 60
        print(f'Running OpenAI cross vendor judge ({OPENAI_JUDGE_MODEL})')
        print(f'  {len(df)} calls. estimated cost: ${est_cost:.3f}. '
              f'runtime at {REQUEST_DELAY_S}s delay: ~{est_min:.0f} min')

    oai_rows = []
    total_in = 0
    total_out = 0
    MAX_RETRIES = 3

    for _, row in df.iterrows():
        prompt = build_judge_prompt(row.to_dict(), row['xai_model'], row['instance_id'])
        scores, in_tok, out_tok, raw = {}, 0, 0, ''
        for attempt in range(MAX_RETRIES):
            response = ask_openai_text(
                prompt,
                system=JUDGE_SYSTEM,
                model=OPENAI_JUDGE_MODEL,
                max_tokens=900,
                request_delay_s=REQUEST_DELAY_S if attempt == 0 else 5,
            )
            usage   = response.get('usage', {})
            in_tok  = usage.get('input_tokens',  0)
            out_tok = usage.get('output_tokens', 0)
            raw     = response['content'][0]['text'].strip()
            scores  = _parse_judge_response(raw)
            if all(scores.get(k) is not None for k in ['faithfulness', 'clarity', 'completeness']):
                break
            print(f'Retry {attempt+1}/{MAX_RETRIES}: '
                  f'{row["pipeline_label"]} {row["xai_model"]} inst={row["instance_id"]}')

        total_in  += in_tok
        total_out += out_tok
        oai_rows.append({
            'pipeline_label': row['pipeline_label'],
            'xai_model':      row['xai_model'],
            'instance_id':    row['instance_id'],
            'faithfulness':   scores.get('faithfulness',  None),
            'clarity':        scores.get('clarity',       None),
            'completeness':   scores.get('completeness',  None),
            'reasoning': {
                'faithfulness': scores.get('faithfulness_reasoning', ''),
                'clarity':      scores.get('clarity_reasoning',      ''),
                'completeness': scores.get('completeness_reasoning', ''),
            },
            'raw_response': raw,
        })
        cost_call = (in_tok * OAI_COST_IN_PER_M + out_tok * OAI_COST_OUT_PER_M) / 1_000_000
        print(f'  {row["pipeline_label"]:12s} {row["xai_model"]} inst={row["instance_id"]:4d}  '
              f'F={scores.get("faithfulness","?")} '
              f'C={scores.get("clarity","?")} '
              f'Co={scores.get("completeness","?")}  '
              f'in={in_tok}  ${cost_call:.5f}')

    OPENAI_CACHE_PATH.write_text(json.dumps(oai_rows, indent=2, ensure_ascii=False))
    total_cost = (total_in * OAI_COST_IN_PER_M + total_out * OAI_COST_OUT_PER_M) / 1_000_000
    print(f'\nTotal: input={total_in}  output={total_out}  cost: ${total_cost:.4f}')

# Evaluation
judge_oai_df = pd.DataFrame(json.loads(OPENAI_CACHE_PATH.read_text()))
for col in ['faithfulness', 'clarity', 'completeness']:
    judge_oai_df[col] = pd.to_numeric(judge_oai_df[col], errors='coerce')

null_mask = judge_oai_df[['faithfulness', 'clarity', 'completeness']].isnull().any(axis=1)
if null_mask.any():
    print(f'WARNING: {null_mask.sum()} entries with None scores (parsing failed):')
    display(judge_oai_df[null_mask][['pipeline_label', 'xai_model', 'instance_id']])

oai_summary = judge_oai_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].mean().round(2)
oai_std     = judge_oai_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].std().round(3)
oai_n       = judge_oai_df.groupby('pipeline_label')[['faithfulness']].count().rename(columns={'faithfulness': 'n'})
oai_std.columns = ['faith_std', 'clarity_std', 'complete_std']

tv = judge_oai_df[['faithfulness', 'clarity', 'completeness']].count().sum()
t5 = (judge_oai_df[['faithfulness', 'clarity', 'completeness']] == 5).sum().sum()
t4 = (judge_oai_df[['faithfulness', 'clarity', 'completeness']] == 4).sum().sum()

print(f'\nOpenAI {OPENAI_JUDGE_MODEL} cross vendor judge scores (identical rubric):')
display(oai_summary.join(oai_std).join(oai_n))
print(f'\nCeiling: 5: {t5} ({100*t5/tv:.0f}%), 4: {t4} ({100*t4/tv:.0f}%)')

Running OpenAI cross vendor judge (gpt-4o-mini)
  80 calls. estimated cost: $0.032. runtime at 20s delay: ~27 min
  Template     XGB inst= 224  F=5 C=5 Co=5  in=2355  $0.00047
  Template     XGB inst= 580  F=5 C=5 Co=5  in=2354  $0.00046
  Template     XGB inst=1041  F=5 C=5 Co=5  in=2351  $0.00045
  Template     XGB inst=1481  F=4 C=4 Co=4  in=2349  $0.00048
  Template     XGB inst=1677  F=4 C=5 Co=5  in=2351  $0.00049
  Template     XGB inst=2058  F=5 C=5 Co=5  in=2351  $0.00047
  Template     XGB inst=2510  F=3 C=3 Co=4  in=2351  $0.00052
  Template     XGB inst=3543  F=4 C=4 Co=5  in=2350  $0.00047
  Template     XGB inst=3847  F=4 C=5 Co=5  in=2352  $0.00048
  Template     XGB inst=4454  F=5 C=5 Co=5  in=2350  $0.00045
  Template     EBM inst= 224  F=5 C=5 Co=5  in=2355  $0.00047
  Template     EBM inst= 580  F=5 C=5 Co=5  in=2356  $0.00046
  Template     EBM inst=1041  F=5 C=5 Co=5  in=2352  $0.00045
  Template     EBM inst=1481  F=4 C=4 Co=5  in=2349  $0.00047
  Template     EBM

,faithfulness,clarity,completeness,faith_std,clarity_std,complete_std,n
pipeline_label,,,,,,,
JSON to Text,4.25,4.10,4.85,0.851,0.641,0.489,20
Template,4.55,4.65,4.85,0.686,0.587,0.366,20
Tool Use,4.75,4.25,4.95,0.550,0.716,0.224,20
Vision,4.40,4.00,4.95,0.821,0.725,0.224,20



Ceiling: 5: 156 (65%), 4: 59 (25%)


In [18]:
# Ranking stability comparison: Opus vs OpenAI cross vendor
# Core question: does the pipeline ranking stay stable when a different model scores?

labels_ord = list(PIPELINE_COLORS.keys())   # ['Template', 'JSON to Text', 'Vision', 'Tool Use']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
judge_pair = [
    ('Opus\n(claude-opus-4-8)',       judge_opus_df, '#4c72b0', '#1a3a6e'),
    (f'OpenAI\n{OPENAI_JUDGE_MODEL}',  judge_oai_df,  '#e87d3e', '#a34e00'),
]

for ax_i, (crit, title) in enumerate([
    ('faithfulness',  'Faithfulness'),
    ('clarity',       'Clarity'),
    ('completeness',  'Completeness'),
]):
    x = np.arange(len(labels_ord)); w = 0.35
    for j, (vlabel, df_, face, edge) in enumerate(judge_pair):
        vals = [df_[df_.pipeline_label == l][crit].mean() for l in labels_ord]
        bars = axes[ax_i].bar(x + (j - 0.5) * w, vals, w, label=vlabel,
                               color=face, edgecolor=edge, linewidth=0.8)
        for bar in bars:
            h = bar.get_height()
            if not np.isnan(h):
                axes[ax_i].text(bar.get_x() + bar.get_width() / 2, h + 0.06,
                                f'{h:.2f}', ha='center', fontsize=7.5)
    axes[ax_i].set_xticks(x); axes[ax_i].set_xticklabels(labels_ord, fontsize=9)
    axes[ax_i].set_ylim(0, 5.9); axes[ax_i].set_ylabel('Mean score (1 to 5)', fontsize=9)
    axes[ax_i].set_title(title, fontsize=11)
    axes[ax_i].axhline(4, color='#aaa', linestyle='--', linewidth=0.7, alpha=0.6)
    if ax_i == 0:
        axes[ax_i].legend(fontsize=8, loc='lower right')

plt.suptitle(
    f'Cross vendor judge: Anthropic Opus  vs.  OpenAI {OPENAI_JUDGE_MODEL}  (identical rubric)',
    y=1.02, fontsize=10,
)
plt.tight_layout()
out = RESULTS_DIR / 'eval_judge_cross_vendor.png'
plt.savefig(out, dpi=130, bbox_inches='tight')
display(fig)
print(f'Saved: {out}')

# Rank correlation (Spearman) for ranking stability
from scipy.stats import spearmanr

print('\n-- Ranking stability: Spearman rank coefficient between Opus and OpenAI --')
print('(basis: pipeline rank by mean score per criterion, n=4 pipelines)\n')
for metric in ['faithfulness', 'clarity', 'completeness']:
    opus_means = judge_opus_df.groupby('pipeline_label')[metric].mean().reindex(labels_ord)
    oai_means  = judge_oai_df.groupby('pipeline_label')[metric].mean().reindex(labels_ord)
    rho, pval  = spearmanr(opus_means.values, oai_means.values)
    print(f'  {metric.capitalize():<14}: rho = {rho:.3f}  (p = {pval:.3f})')

print()
print('Interpretation: rho close to 1 means the ranking is stable. rho below 0.6 means')
print('diverging orders, so a self preference effect could be visible.')
print()

# Total ranking of both judges side by side
print('-- Total ranking (sum of faithfulness + clarity + completeness) --')
opus_total = judge_opus_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].mean().sum(axis=1)
oai_total  = judge_oai_df.groupby('pipeline_label')[['faithfulness', 'clarity', 'completeness']].mean().sum(axis=1)
rank_df = pd.DataFrame({
    'Opus_Score':      opus_total.round(2),
    'Opus_Rank':       opus_total.rank(ascending=False).astype(int),
    f'OAI_{OPENAI_JUDGE_MODEL}_Score': oai_total.round(2),
    f'OAI_{OPENAI_JUDGE_MODEL}_Rank':  oai_total.rank(ascending=False).astype(int),
}).loc[labels_ord]
display(rank_df)

# Save results as CSV
oai_stat_rows = []
for pipeline in labels_ord:
    subset = judge_oai_df[judge_oai_df['pipeline_label'] == pipeline]
    entry  = {'pipeline': pipeline, 'judge': f'openai_{OPENAI_JUDGE_MODEL}', 'n': len(subset)}
    for metric in ['faithfulness', 'clarity', 'completeness']:
        lo, hi, obs = bootstrap_ci(subset[metric].values)
        entry[f'{metric}_mean']  = round(obs, 3)
        entry[f'{metric}_ci_lo'] = round(lo,  3)
        entry[f'{metric}_ci_hi'] = round(hi,  3)
    oai_stat_rows.append(entry)

oai_stat_df = pd.DataFrame(oai_stat_rows)
out_csv = RESULTS_DIR / 'eval_judge_cross_vendor.csv'
oai_stat_df.to_csv(out_csv, index=False)
print(f'\nStatistics saved: {out_csv}')
print(f'Plot saved:      {out}')

<Figure size 1500x500 with 3 Axes>

Saved: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/eval_judge_cross_vendor.png

-- Ranking stability: Spearman rank coefficient between Opus and OpenAI --
(basis: pipeline rank by mean score per criterion, n=4 pipelines)

  Faithfulness  : rho = 0.258  (p = 0.742)
  Clarity       : rho = 0.949  (p = 0.051)
  Completeness  : rho = 0.577  (p = 0.423)

Interpretation: rho close to 1 means the ranking is stable. rho below 0.6 means
diverging orders, so a self preference effect could be visible.

-- Total ranking (sum of faithfulness + clarity + completeness) --


,Opus_Score,Opus_Rank,OAI_gpt-4o-mini_Score,OAI_gpt-4o-mini_Rank
pipeline_label,,,,
Template,13.95,3,14.05,1
JSON to Text,14.10,1,13.20,4
Vision,13.50,4,13.35,3
Tool Use,14.10,1,13.95,2



Statistics saved: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/eval_judge_cross_vendor.csv
Plot saved:      /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/eval_judge_cross_vendor.png


## 9. Inter judge agreement: Krippendorff's alpha and Kendall's tau

Which judges are paired?

| Judge | Model | Sample |
|---|---|---|
| Primary | Opus (claude-opus-4-8) | canonical (10 inst x 4 pipelines x 2 XAI = 80) |
| Cross vendor | OpenAI (gpt-4o-mini) | canonical |

Metrics:
* Krippendorff's alpha (interval metric): agreement across both raters at once.
  alpha < 0.20 = weak, 0.20 to 0.60 = moderate, > 0.60 = good (Krippendorff 2011).
* Pairwise Kendall's tau: rank correlation of the 80 single scores between the two judges.
  Does the order (which instance is better) agree across judges?

In [19]:
import numpy as np
from scipy.stats import kendalltau

# Helper functions

def krippendorff_alpha_interval(reliability_data: np.ndarray) -> float:
    """Krippendorff's alpha with interval metric (squared differences).

    Parameters
    ----------
    reliability_data : ndarray shape (n_raters, n_units)
                       NaN = missing value

    Returns
    -------
    float : alpha in [-1, 1]; 1.0 = perfect agreement, 0.0 = chance.

    Notes
    -----
    The interval metric is the standard approximation for ordinal Likert data (1 to 5).
    Formula: Hayes and Krippendorff (2007) Communication Methods and Measures.
    """
    data = np.asarray(reliability_data, dtype=float)  # (n_raters, n_units)
    n_raters, n_units = data.shape

    # Observed disagreement: average squared diff across all rater pairs per unit
    D_o = 0.0
    n_pairs_total = 0
    for u in range(n_units):
        col = data[:, u]
        valid = col[~np.isnan(col)]
        m_u = len(valid)
        if m_u < 2:
            continue
        for i in range(m_u):
            for j in range(i + 1, m_u):
                D_o += (valid[i] - valid[j]) ** 2
                n_pairs_total += 1

    if n_pairs_total == 0:
        return np.nan
    D_o /= n_pairs_total

    # Expected disagreement: across all values ignoring unit structure
    all_values = data[~np.isnan(data)]
    n = len(all_values)
    if n < 2:
        return np.nan
    D_e = 0.0
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            D_e += (all_values[i] - all_values[j]) ** 2
            count += 1
    D_e /= count

    if D_e == 0:
        return 1.0 if D_o == 0 else np.nan

    return 1.0 - D_o / D_e


def _align_scores(df_a, df_b, metric,
                  id_cols=('pipeline_label', 'xai_model', 'instance_id')):
    """Return two aligned score vectors (missing pairs excluded)."""
    a = df_a.set_index(list(id_cols))[metric]
    b = df_b.set_index(list(id_cols))[metric]
    common = a.index.intersection(b.index)
    xa, xb = a.loc[common].values.astype(float), b.loc[common].values.astype(float)
    mask = ~(np.isnan(xa) | np.isnan(xb))
    return xa[mask], xb[mask]


# Collect the two judge DataFrames (Opus primary, OpenAI cross vendor).
_judge_versions = {
    'Opus (claude-opus-4-8)':         judge_opus_df,
    f'OpenAI ({OPENAI_JUDGE_MODEL})': judge_oai_df,
}

print(f'Judges loaded: {list(_judge_versions.keys())}')
print(f'Entries per judge: { {k: len(v) for k, v in _judge_versions.items()} }')

Judges loaded: ['Opus (claude-opus-4-8)', 'OpenAI (gpt-4o-mini)']
Entries per judge: {'Opus (claude-opus-4-8)': 80, 'OpenAI (gpt-4o-mini)': 80}


In [20]:
# Krippendorff's alpha over the paired judges
judge_labels = list(_judge_versions.keys())
id_cols = ['pipeline_label', 'xai_model', 'instance_id']

# Common units = intersection over all judges
_all_indices = [
    set(zip(df_[id_cols[0]], df_[id_cols[1]], df_[id_cols[2]]))
    for df_ in _judge_versions.values()
]
_common_units = sorted(_all_indices[0].intersection(*_all_indices[1:]))
print(f'Common units for alpha: {len(_common_units)} '
      f'(of max {len(df)}, overlap of all judges)\n')

alpha_rows = []
for metric in ['faithfulness', 'clarity', 'completeness']:
    reliability = []
    for df_ in _judge_versions.values():
        idx = df_.set_index(id_cols)[metric]
        row_scores = []
        for unit in _common_units:
            key = (unit[0], unit[1], unit[2])
            row_scores.append(float(idx.get(key, np.nan)))
        reliability.append(row_scores)

    alpha = krippendorff_alpha_interval(np.array(reliability))
    alpha_rows.append({'metric': metric.capitalize(), 'krippendorff_alpha': round(alpha, 3)})
    print(f'  Krippendorff alpha ({metric:14s}): {alpha:.3f}')

alpha_df = pd.DataFrame(alpha_rows).set_index('metric')
print()
print('Guideline: alpha < 0.20 = weak, 0.20 to 0.60 = moderate, > 0.60 = good (Krippendorff 2011)')

Common units for alpha: 80 (of max 80, overlap of all judges)

  Krippendorff alpha (faithfulness  ): -0.093
  Krippendorff alpha (clarity       ): 0.074
  Krippendorff alpha (completeness  ): -0.062

Guideline: alpha < 0.20 = weak, 0.20 to 0.60 = moderate, > 0.60 = good (Krippendorff 2011)


In [21]:
# Pairwise Kendall's tau between judges
print("Pairwise Kendall's tau (rank correlation on single score level, n=80 units)")
print('tau > 0.6 = good agreement of the instance order\n')

tau_tables = {}
for metric in ['faithfulness', 'clarity', 'completeness']:
    rows = []
    for i, (la, dfa) in enumerate(list(_judge_versions.items())):
        for lb, dfb in list(_judge_versions.items())[i+1:]:
            xa, xb = _align_scores(dfa, dfb, metric)
            if len(xa) < 3:
                rows.append({'Judge A': la.replace('\n', ' '),
                             'Judge B': lb.replace('\n', ' '),
                             'n': len(xa), 'tau': np.nan, 'p': np.nan})
                continue
            tau, pval = kendalltau(xa, xb)
            rows.append({'Judge A': la.replace('\n', ' '),
                         'Judge B': lb.replace('\n', ' '),
                         'n': len(xa),
                         'tau': round(tau, 3),
                         'p':   round(pval, 4)})
    tau_df = pd.DataFrame(rows)
    tau_tables[metric] = tau_df
    print(f'-- {metric.upper()} --')
    display(tau_df.to_string(index=False))
    print()

# Summary and CSV export
agreement_summary = pd.concat(
    [df_.assign(metric=m) for m, df_ in tau_tables.items()],
    ignore_index=True,
)[['metric', 'Judge A', 'Judge B', 'n', 'tau', 'p']]
out_agr = RESULTS_DIR / 'eval_inter_judge_agreement.csv'
agreement_summary.to_csv(out_agr, index=False)

alpha_out = RESULTS_DIR / 'eval_krippendorff_alpha.csv'
alpha_df.to_csv(alpha_out)

print(f'Saved: {out_agr}')
print(f'Saved: {alpha_out}')
print()
print('=== SUMMARY ===')
print()
print('Krippendorff alpha over both judges:')
display(alpha_df)
print()
print('Interpretation for the paper:')
print('  High tau (> 0.6) between the judges means the ranking statements are robust.')
print('  Low alpha means the judges agree less on the absolute score than on the order.')
print('  The vendor change (OpenAI vs Anthropic) affects absolute scores more than the')
print('  relative instance order.')

Pairwise Kendall's tau (rank correlation on single score level, n=80 units)
tau > 0.6 = good agreement of the instance order

-- FAITHFULNESS --


'               Judge A              Judge B  n   tau      p\nOpus (claude-opus-4-8) OpenAI (gpt-4o-mini) 80 0.061 0.5669'


-- CLARITY --


'               Judge A              Judge B  n   tau      p\nOpus (claude-opus-4-8) OpenAI (gpt-4o-mini) 80 0.087 0.4116'


-- COMPLETENESS --


'               Judge A              Judge B  n    tau     p\nOpus (claude-opus-4-8) OpenAI (gpt-4o-mini) 80 -0.071 0.528'


Saved: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/eval_inter_judge_agreement.csv
Saved: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/eval_krippendorff_alpha.csv

=== SUMMARY ===

Krippendorff alpha over both judges:


,krippendorff_alpha
metric,
Faithfulness,-0.093
Clarity,0.074
Completeness,-0.062



Interpretation for the paper:
  High tau (> 0.6) between the judges means the ranking statements are robust.
  Low alpha means the judges agree less on the absolute score than on the order.
  The vendor change (OpenAI vs Anthropic) affects absolute scores more than the
  relative instance order.


## 10. Judge sensitivity: ranking stability across both judges

The pipeline ranking is the actual statement of the paper, not the absolute scores.
This section checks whether the order (1st to 4th place) stays stable even when the
absolute scores move.

Procedure:
1. Both judge scores side by side (mean per pipeline x criterion)
2. Rank per judge and criterion (1 = best pipeline)
3. Spearman rho between the judges on rank level (n = 4 pipelines)
4. Explicit stability statement for the paper

In [22]:
from scipy.stats import spearmanr

METRICS   = ['faithfulness', 'clarity', 'completeness']
PIP_ORDER = list(PIPELINE_COLORS.keys())   # ['Template', 'JSON to Text', 'Vision', 'Tool Use']

# Mean score table
score_cols = {}
for jlabel, jdf in _judge_versions.items():
    short = jlabel
    for metric in METRICS:
        means = jdf.groupby('pipeline_label')[metric].mean().reindex(PIP_ORDER)
        score_cols[f'{metric[:5].capitalize()}\n{short}'] = means.values

score_table = pd.DataFrame(score_cols, index=PIP_ORDER)
print('Mean judge scores of both judges side by side (1 to 5):')
display(score_table.round(2))

# Rank table (1 = best)
rank_cols = {}
for jlabel, jdf in _judge_versions.items():
    short = jlabel
    for metric in METRICS:
        means = jdf.groupby('pipeline_label')[metric].mean().reindex(PIP_ORDER)
        ranks = means.rank(ascending=False, method='min').astype(int)
        rank_cols[f'{metric[:5].capitalize()}\n{short}'] = ranks.values

rank_table = pd.DataFrame(rank_cols, index=PIP_ORDER)
print('\nRanking per judge and criterion (1 = best pipeline):')
display(rank_table)

Mean judge scores of both judges side by side (1 to 5):


,Faith\nOpus (claude-opus-4-8),Clari\nOpus (claude-opus-4-8),Compl\nOpus (claude-opus-4-8),Faith\nOpenAI (gpt-4o-mini),Clari\nOpenAI (gpt-4o-mini),Compl\nOpenAI (gpt-4o-mini)
Template,5.0,4.15,4.8,4.55,4.65,4.85
JSON to Text,5.0,4.10,5.0,4.25,4.10,4.85
Vision,4.5,4.00,5.0,4.40,4.00,4.95
Tool Use,5.0,4.10,5.0,4.75,4.25,4.95



Ranking per judge and criterion (1 = best pipeline):


,Faith\nOpus (claude-opus-4-8),Clari\nOpus (claude-opus-4-8),Compl\nOpus (claude-opus-4-8),Faith\nOpenAI (gpt-4o-mini),Clari\nOpenAI (gpt-4o-mini),Compl\nOpenAI (gpt-4o-mini)
Template,1,1,4,2,1,3
JSON to Text,1,2,1,4,3,3
Vision,4,4,1,3,4,1
Tool Use,1,2,1,1,2,1


In [23]:
# Spearman rho between the judges (on pipeline rank level)
# Basis: total rank = sum of the three criteria scores per pipeline (one rank vector per judge)
print('Spearman rho between judges (basis: pipeline total rank, n=4 pipelines)')
print('Note: at n=4 p values are not meaningful, the rho pattern is what counts.\n')

judge_list  = list(_judge_versions.items())
n_j         = len(judge_list)
rho_matrix  = np.full((n_j, n_j), np.nan)
short_labels = [jl.replace('\n', ' ') for jl, _ in judge_list]

# Total score per judge x pipeline (sum over the three criteria)
def _pipeline_totals(jdf):
    return jdf.groupby('pipeline_label')[METRICS].mean().sum(axis=1).reindex(PIP_ORDER).values

for i, (la, dfa) in enumerate(judge_list):
    for j, (lb, dfb) in enumerate(judge_list):
        if i == j:
            rho_matrix[i, j] = 1.0
            continue
        rho, _ = spearmanr(_pipeline_totals(dfa), _pipeline_totals(dfb))
        rho_matrix[i, j] = round(rho, 3)

rho_df = pd.DataFrame(rho_matrix, index=short_labels, columns=short_labels)
display(rho_df)

# Visualisation: total ranking of both judges
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(PIP_ORDER))
w = 0.8 / n_j
colors_jv = ['#4c72b0', '#e87d3e', '#55a868', '#888888']

for i, (jlabel, jdf) in enumerate(judge_list):
    totals = jdf.groupby('pipeline_label')[METRICS].mean().sum(axis=1).reindex(PIP_ORDER)
    bars = ax.bar(x + (i - n_j/2 + 0.5) * w, totals.values, w,
                  label=short_labels[i], color=colors_jv[i % len(colors_jv)],
                  edgecolor='white', linewidth=0.6)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.03,
                f'{h:.1f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x); ax.set_xticklabels(PIP_ORDER, fontsize=10)
ax.set_ylabel('Total score (sum of faithfulness + clarity + completeness, max=15)', fontsize=9)
ax.set_ylim(0, 16.5)
ax.axhline(12, color='#ccc', linestyle='--', linewidth=0.7)
ax.legend(fontsize=8, loc='lower right')
ax.set_title('Pipeline total ranking across both judges', fontsize=11)
plt.tight_layout()
out_sens = RESULTS_DIR / 'eval_judge_sensitivity.png'
plt.savefig(out_sens, dpi=130, bbox_inches='tight')
display(fig)
print(f'Saved: {out_sens}')

Spearman rho between judges (basis: pipeline total rank, n=4 pipelines)
Note: at n=4 p values are not meaningful, the rho pattern is what counts.



,Opus (claude-opus-4-8),OpenAI (gpt-4o-mini)
Opus (claude-opus-4-8),1.000,-0.211
OpenAI (gpt-4o-mini),-0.211,1.000


<Figure size 900x450 with 1 Axes>

Saved: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/eval_judge_sensitivity.png


In [24]:
# Explicit stability statement
print('=== JUDGE SENSITIVITY: STABILITY STATEMENT ===')
print()

# Determine the ranking from the total score per judge
rank_by_judge = {}
for jlabel, jdf in _judge_versions.items():
    totals = jdf.groupby('pipeline_label')[METRICS].mean().sum(axis=1).reindex(PIP_ORDER)
    ranked = totals.rank(ascending=False, method='min').astype(int)
    rank_by_judge[jlabel.replace('\n', ' ')] = ranked

rank_overview = pd.DataFrame(rank_by_judge, index=PIP_ORDER)
print('Total rank (1 = best pipeline) per judge:')
display(rank_overview)
print()

# First and last place per judge
first_place = {judge: ranks.idxmin() for judge, ranks in rank_by_judge.items()}
last_place  = {judge: ranks.idxmax() for judge, ranks in rank_by_judge.items()}

first_set = set(first_place.values())
last_set  = set(last_place.values())

print(f'First place pipeline per judge: {first_place}')
print(f'Last place pipeline per judge: {last_place}')
print()

# Mean Spearman rank coefficient over all off diagonal pairs
off_diag = [rho_matrix[i,j] for i in range(n_j) for j in range(n_j) if i != j]
mean_rho = np.nanmean(off_diag)

print(f'Mean pairwise Spearman rho (pipeline rank): {mean_rho:.3f}')
print()

# Stability verdict
if mean_rho >= 0.8 and len(first_set) == 1:
    stability = 'STABLE'
    detail = (f'First place consistent: {list(first_set)[0]}. '
              f'mean Spearman rho = {mean_rho:.2f}, high rank agreement across judges.')
elif mean_rho >= 0.6:
    stability = 'MOSTLY STABLE'
    detail = (f'First place in {len(first_set)} variant(s): {first_set}. '
              f'mean Spearman rho = {mean_rho:.2f}, slight differences, base order preserved.')
else:
    stability = 'UNSTABLE'
    detail = (f'First place varies: {first_set}. '
              f'mean Spearman rho = {mean_rho:.2f}, judge choice changes the order substantially.')

print(f'Stability verdict: {stability}')
print(f'Detail: {detail}')
print()
print('Note for the paper (n=4 pipelines):')
print('  Spearman rho on n=4 points has no inferential power (p is always > 0.05).')
print('  The pattern (rho near 1 or clearly below) is still interpretable.')

# Save CSV
rank_overview.to_csv(RESULTS_DIR / 'eval_judge_sensitivity_ranks.csv')
score_table.round(3).to_csv(RESULTS_DIR / 'eval_judge_sensitivity_scores.csv')
print(f'\nSaved: {RESULTS_DIR}/eval_judge_sensitivity_ranks.csv')
print(f'Saved: {RESULTS_DIR}/eval_judge_sensitivity_scores.csv')

=== JUDGE SENSITIVITY: STABILITY STATEMENT ===

Total rank (1 = best pipeline) per judge:


,Opus (claude-opus-4-8),OpenAI (gpt-4o-mini)
Template,3,1
JSON to Text,1,4
Vision,4,3
Tool Use,1,2



First place pipeline per judge: {'Opus (claude-opus-4-8)': 'JSON to Text', 'OpenAI (gpt-4o-mini)': 'Template'}
Last place pipeline per judge: {'Opus (claude-opus-4-8)': 'Vision', 'OpenAI (gpt-4o-mini)': 'JSON to Text'}

Mean pairwise Spearman rho (pipeline rank): -0.211

Stability verdict: UNSTABLE
Detail: First place varies: {'JSON to Text', 'Template'}. mean Spearman rho = -0.21, judge choice changes the order substantially.

Note for the paper (n=4 pipelines):
  Spearman rho on n=4 points has no inferential power (p is always > 0.05).
  The pattern (rho near 1 or clearly below) is still interpretable.

Saved: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/eval_judge_sensitivity_ranks.csv
Saved: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/eval_judge_sensitivity_scores.csv
